# Person 1 Stages 9-10: Final Detector

This notebook reproduces the historical OOF hybrid comparison and the unlabeled Test_Data production path. Hidden Test_Data labels are never loaded or displayed.

In [ ]:
from pathlib import Path
import pandas as pd
from src.dataset_loader import load_training_data, load_test_data
from src.stage8_validation import generate_oof_predictions, load_stage7_best_configuration
from src.final_hybrid_detector import FinalHybridDetector, compare_hybrid_approaches, LOCKED_THRESHOLD

output_dir = Path('../outputs')
training = load_training_data()
test = load_test_data()
config = load_stage7_best_configuration(training)
config

## Historical leakage-free comparison

In [ ]:
oof = generate_oof_predictions(training, model_name=config.model_name, feature_set_name=config.feature_set_name, random_state=42)
comparison = compare_hybrid_approaches(oof, random_state=42)
comparison

The supervised-only approach is retained when soft evidence adds false positives. The threshold remains locked at `0.40`; it is not re-tuned on Test_Data.

In [ ]:
detector = FinalHybridDetector(threshold=LOCKED_THRESHOLD, random_state=42).fit(training)
diagnostics = detector.predict_with_diagnostics(test)
submission = diagnostics[['Test_ID', 'Final_Validity_Label']].rename(columns={'Final_Validity_Label': 'Validity_Label'})
assert len(test) == len(submission) == 350
assert submission['Test_ID'].is_unique
assert set(submission['Validity_Label']).issubset({'Valid', 'Invalid'})
submission['Validity_Label'].value_counts()

In [ ]:
# Diagnostics only: no hidden labels or Test_Data accuracy metrics are computed.
diagnostics[['Test_ID', 'Invalid_Probability', 'Anomaly_Score', 'Final_Validity_Label', 'Reason_Evidence']].head()

In [ ]:
submission.to_csv(output_dir / 'task1_predictions.csv', index=False)
diagnostics.to_csv(output_dir / 'p1_stage10_test_predictions_diagnostics.csv', index=False)